In [2]:
import pandas as pd

pd.set_option("display.max_colwidth", None)

In [3]:
replacement_record = pd.read_excel(
    "2024-2025-LSLR-Data.xlsx",
    header=2
)

lead_service_line_inventory = pd.read_excel(
    "DSMI-Service-Line-Materials-Estimates.xlsx",
    header=2
)

In [4]:
replacement_record.head()

,Public Water Supply ID,Supply Name,County,Year,Number of Lead and GPCL Service Lines Replaced
0,MI0000040,ADRIAN,LENAWEE,2021,57
1,MI0000120,ALLEGAN,ALLEGAN,2021,12
2,MI0000130,ALLEN PARK,WAYNE,2021,8
3,MI0000150,"ALMONT, VILLAGE OF",LAPEER,2021,7
4,MI0000160,"ALPENA, CITY OF",ALPENA,2021,25


In [5]:
replacement_record.columns = [
    "pwsid",
    "supply_name",
    "county",
    "year",
    "lines_replaced"
]

In [6]:
lead_service_line_inventory.head()

,Public Water Supply ID,Supply Name,Lead,Galvanized Previously Connected to Lead,Non-Lead,Lead Status Unknown,Total Service Lines
0,MI0000011,ACME TOWNSHIP - HOPE VILLAGE,0,0.0,1.0,0.0,1.0
1,MI0000012,ADA TOWNSHIP,0,0.0,2431.0,0.0,2431.0
2,MI0000020,ADAMS TOWNSHIP,0,0.0,481.0,154.0,635.0
3,MI0000030,ADDISON,0,0.0,118.0,NaN,152.0
4,MI0000040,ADRIAN,5,2280.0,3876.0,0.0,6161.0


In [7]:
lead_service_line_inventory.columns = [
    "pwsid",
    "supply_name",
    "lead_lines",
    "gpcl_lines",
    "non_lead_lines",
    "unknown_lines",
    "total_lines"
]

In [8]:
lead_service_line_inventory["pwsid"].duplicated().sum()

np.int64(0)

In [9]:
replacement_record.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 916 entries, 0 to 915
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   pwsid           916 non-null    object
 1   supply_name     916 non-null    object
 2   county          916 non-null    object
 3   year            916 non-null    int64 
 4   lines_replaced  916 non-null    int64 
dtypes: int64(2), object(3)
memory usage: 35.9+ KB


In [10]:
replacement_record.duplicated(["pwsid", "year"]).sum()

np.int64(2)

In [11]:
replacement_record["year"].unique()

array([2021, 2022, 2023, 2024])

In [12]:
replacement_record["lines_replaced"].describe()

count     916.000000
mean       76.332969
std       328.649679
min         1.000000
25%         4.000000
50%        10.000000
75%        45.000000
max      7468.000000
Name: lines_replaced, dtype: float64

In [13]:
duplicate_rows = replacement_record[
    replacement_record.duplicated(["pwsid", "year"], keep=False)
]

duplicate_rows

,pwsid,supply_name,county,year,lines_replaced
322,MI0003525,KALAMAZOO LAKE SEWER & WATER AUTHORITY- City of Douglas,ALLEGAN,2022,7
323,MI0003525,KALAMAZOO LAKE SEWER & WATER AUTHORITY - City of Saugatuck,ALLEGAN,2022,6
541,MI0003525,KALAMAZOO LAKE SEWER & WATER AUTHORITY - City of Douglas,ALLEGAN,2023,17
542,MI0003525,KALAMAZOO LAKE SEWER & WATER AUTHORITY - KLSWA,ALLEGAN,2023,1


Keep all data

In [14]:
replacement_clean = (
    replacement_record
    .groupby(["pwsid", "year", "county"], as_index=False)["lines_replaced"]
    .sum()
)

In [15]:
replacement_clean = replacement_clean.rename(
    columns={"county": "county_for_checking"}
)

In [16]:
replacement_clean.duplicated(["pwsid", "year"]).sum()

np.int64(0)

In [17]:
replacement_clean

,pwsid,year,county_for_checking,lines_replaced
0,MI0000040,2021,LENAWEE,57
1,MI0000040,2022,LENAWEE,124
2,MI0000040,2023,LENAWEE,58
3,MI0000040,2024,LENAWEE,88
4,MI0000100,2023,CALHOUN,4
...,...,...,...,...
909,MI0007260,2024,WASHTENAW,22
910,MI0007270,2021,OTTAWA,81
911,MI0007270,2022,OTTAWA,138
912,MI0007270,2023,OTTAWA,246


In [18]:
replacement_clean.to_csv(
    "clean_LSLR.csv",
    index=False
)

In [19]:
lead_service_line_inventory.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1383 entries, 0 to 1382
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   pwsid           1383 non-null   object 
 1   supply_name     1383 non-null   object 
 2   lead_lines      1226 non-null   object 
 3   gpcl_lines      1217 non-null   float64
 4   non_lead_lines  1325 non-null   float64
 5   unknown_lines   1180 non-null   float64
 6   total_lines     1319 non-null   float64
dtypes: float64(4), object(3)
memory usage: 75.8+ KB


In [20]:
inventory_missing_rows = lead_service_line_inventory[
    lead_service_line_inventory[
        ["lead_lines", "gpcl_lines", "non_lead_lines", "unknown_lines", "total_lines"]
    ].isna().any(axis=1)
]

inventory_missing_rows.head(20)

,pwsid,supply_name,lead_lines,gpcl_lines,non_lead_lines,unknown_lines,total_lines
3,MI0000030,ADDISON,0,0.0,118.0,NaN,152.0
11,MI0000088,ALBEE TOWNSHIP,NaN,0.0,203.0,0.0,203.0
24,MI0000200,HEMATITE TOWNSHIP,0,0.0,NaN,0.0,0.0
26,MI0000215,ANDELINA FARMS,0,0.0,289.0,NaN,289.0
35,MI0000280,"AU GRES, CITY OF",NaN,NaN,582.0,NaN,NaN
41,MI0000322,AUSTIN COMMONS II,0,0.0,112.0,NaN,112.0
48,MI0000370,"BANCROFT, VILLAGE OF",NaN,NaN,126.0,71.0,197.0
55,MI0000426,BARRY TOWNSHIP,NaN,NaN,101.0,NaN,101.0
64,MI0000501,BEACH HOUSE APARTMENTS,NaN,0.0,17.0,NaN,NaN
67,MI0000508,BEAR CREEK VILLA,NaN,NaN,21.0,NaN,21.0


Not remove any NaN value

Check 2019

In [21]:
lead_service_line_inventory[
    pd.to_numeric(lead_service_line_inventory["lead_lines"], errors="coerce").isna()
    & lead_service_line_inventory["lead_lines"].notna()
][["pwsid","supply_name","lead_lines"]]

,pwsid,supply_name,lead_lines
79,MI0000580,BELLEVILLE,Not Received
233,MI0001692,MACK. CO. HOUSING-CURTIS,Not Received
241,MI0001740,DEARBORN HEIGHTS,Not Received
295,MI0002145,MACK. CO. HOUSING-ENGADINE,Not Received
316,MI0002292,EASTERN MICHIGAN NAZARENE DISTRICT CENTE,Not Received
382,MI0002838,GREAT LAKES WATER AUTHORITY,Not Received
400,MI0002943,HALE HOMESTEAD APARTMENTS,Not Received
426,MI0003117,HERITAGE APARTMENTS,Not Received
430,MI0003131,HIDDEN GLEN APARTMENTS,Not Received
444,MI0003200,"HOLLY, VILLAGE OF",Not Received


In [22]:
cols = [
    "lead_lines",
    "gpcl_lines",
    "non_lead_lines",
    "unknown_lines",
    "total_lines"
]

lead_service_line_inventory[cols] = \
lead_service_line_inventory[cols].apply(
    pd.to_numeric,
    errors="coerce"
)

In [23]:
inventory_client = lead_service_line_inventory.copy()

In [38]:
inventory_client["calc_total"] = (
    inventory_client["lead_lines"].fillna(0)
    + inventory_client["gpcl_lines"].fillna(0)
    + inventory_client["non_lead_lines"].fillna(0)
    + inventory_client["unknown_lines"].fillna(0)
)

inventory_client["inventory_status"] = "complete"

mask_incomplete = inventory_client[
    ["lead_lines", "gpcl_lines", "non_lead_lines", "unknown_lines", "total_lines"]
].isna().any(axis=1)

inventory_client.loc[mask_incomplete, "inventory_status"] = "incomplete"

inventory_client["total_mismatch"] = False

mask_total_check = (
    inventory_client["lead_lines"].notna()
    & inventory_client["gpcl_lines"].notna()
    & inventory_client["non_lead_lines"].notna()
    & inventory_client["unknown_lines"].notna()
    & inventory_client["total_lines"].notna()
)

inventory_client.loc[mask_total_check, "total_mismatch"] = (
    inventory_client.loc[mask_total_check, "calc_total"]
    != inventory_client.loc[mask_total_check, "total_lines"]
)

In [40]:
inventory_client.to_csv(
    "clean_inventory_without_calculation.csv",
    index=False
)

In [24]:
mask = (
    lead_service_line_inventory["unknown_lines"].isna()
    & lead_service_line_inventory["total_lines"].notna()
)

lead_service_line_inventory.loc[mask, "unknown_lines"] = (
    lead_service_line_inventory["total_lines"]
    - lead_service_line_inventory["lead_lines"].fillna(0)
    - lead_service_line_inventory["gpcl_lines"].fillna(0)
    - lead_service_line_inventory["non_lead_lines"].fillna(0)
)

In [25]:
lead_service_line_inventory["calc_total"] = (
    lead_service_line_inventory["lead_lines"].fillna(0)
    + lead_service_line_inventory["gpcl_lines"].fillna(0)
    + lead_service_line_inventory["non_lead_lines"].fillna(0)
    + lead_service_line_inventory["unknown_lines"].fillna(0)
)

In [26]:
lead_service_line_inventory[
    lead_service_line_inventory["calc_total"]
    != lead_service_line_inventory["total_lines"]
]

,pwsid,supply_name,lead_lines,gpcl_lines,non_lead_lines,unknown_lines,total_lines,calc_total
35,MI0000280,"AU GRES, CITY OF",NaN,NaN,582.0,NaN,NaN,582.0
64,MI0000501,BEACH HOUSE APARTMENTS,NaN,0.0,17.0,NaN,NaN,17.0
79,MI0000580,BELLEVILLE,NaN,NaN,NaN,NaN,NaN,0.0
95,MI0000716,BINGHAM TOWNSHIP,0.0,0.0,56.0,0.0,NaN,56.0
138,MI0001015,BUTTERCUP SHORES,0.0,0.0,26.0,0.0,70.0,26.0
...,...,...,...,...,...,...,...,...
1316,MI0040609,PINE HAVEN ESTATES MHC,NaN,NaN,NaN,NaN,NaN,0.0
1358,MI0060640,COREWELL HEALTH-BERRIEN CNTR,NaN,NaN,NaN,NaN,NaN,0.0
1366,MI0062841,MEDILODGE OF STERLING,NaN,NaN,3.0,NaN,NaN,3.0
1368,MI0062955,MAJESTIC CARE OF FLUSHING,NaN,NaN,NaN,NaN,NaN,0.0


In [27]:
remaining_problem_rows = lead_service_line_inventory[
    lead_service_line_inventory["calc_total"] != lead_service_line_inventory["total_lines"]
].copy()

remaining_problem_rows["lead_missing"] = remaining_problem_rows["lead_lines"].isna()
remaining_problem_rows["gpcl_missing"] = remaining_problem_rows["gpcl_lines"].isna()
remaining_problem_rows["non_lead_missing"] = remaining_problem_rows["non_lead_lines"].isna()
remaining_problem_rows["unknown_missing"] = remaining_problem_rows["unknown_lines"].isna()
remaining_problem_rows["total_missing"] = remaining_problem_rows["total_lines"].isna()

remaining_problem_rows[
    [
        "pwsid",
        "supply_name",
        "lead_missing",
        "gpcl_missing",
        "non_lead_missing",
        "unknown_missing",
        "total_missing"
    ]
].head(30)

,pwsid,supply_name,lead_missing,gpcl_missing,non_lead_missing,unknown_missing,total_missing
35,MI0000280,"AU GRES, CITY OF",True,True,False,True,True
64,MI0000501,BEACH HOUSE APARTMENTS,True,False,False,True,True
79,MI0000580,BELLEVILLE,True,True,True,True,True
95,MI0000716,BINGHAM TOWNSHIP,False,False,False,False,True
138,MI0001015,BUTTERCUP SHORES,False,False,False,False,False
189,MI0001416,CHOCOLAY BUNGALOWS,False,False,False,False,True
201,MI0001530,COLOMA,True,True,True,True,True
212,MI0001600,CONSTANTINE,False,True,False,False,False
233,MI0001692,MACK. CO. HOUSING-CURTIS,True,True,True,True,True
241,MI0001740,DEARBORN HEIGHTS,True,True,True,True,True


In [28]:
only_total_missing = remaining_problem_rows[
    (~remaining_problem_rows["lead_missing"]) &
    (~remaining_problem_rows["gpcl_missing"]) &
    (~remaining_problem_rows["non_lead_missing"]) &
    (~remaining_problem_rows["unknown_missing"]) &
    (remaining_problem_rows["total_missing"])
]

len(only_total_missing)

8

In [29]:
mask = (
    lead_service_line_inventory["total_lines"].isna() &
    lead_service_line_inventory["lead_lines"].notna() &
    lead_service_line_inventory["gpcl_lines"].notna() &
    lead_service_line_inventory["non_lead_lines"].notna() &
    lead_service_line_inventory["unknown_lines"].notna()
)

lead_service_line_inventory.loc[mask, "total_lines"] = (
    lead_service_line_inventory["lead_lines"]
    + lead_service_line_inventory["gpcl_lines"]
    + lead_service_line_inventory["non_lead_lines"]
    + lead_service_line_inventory["unknown_lines"]
)

In [30]:
lead_service_line_inventory["calc_total"] = (
    lead_service_line_inventory["lead_lines"].fillna(0)
    + lead_service_line_inventory["gpcl_lines"].fillna(0)
    + lead_service_line_inventory["non_lead_lines"].fillna(0)
    + lead_service_line_inventory["unknown_lines"].fillna(0)
)

lead_service_line_inventory[
    lead_service_line_inventory["calc_total"]
    != lead_service_line_inventory["total_lines"]
]

,pwsid,supply_name,lead_lines,gpcl_lines,non_lead_lines,unknown_lines,total_lines,calc_total
35,MI0000280,"AU GRES, CITY OF",NaN,NaN,582.0,NaN,NaN,582.0
64,MI0000501,BEACH HOUSE APARTMENTS,NaN,0.0,17.0,NaN,NaN,17.0
79,MI0000580,BELLEVILLE,NaN,NaN,NaN,NaN,NaN,0.0
138,MI0001015,BUTTERCUP SHORES,0.0,0.0,26.0,0.0,70.0,26.0
201,MI0001530,COLOMA,NaN,NaN,NaN,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...
1316,MI0040609,PINE HAVEN ESTATES MHC,NaN,NaN,NaN,NaN,NaN,0.0
1358,MI0060640,COREWELL HEALTH-BERRIEN CNTR,NaN,NaN,NaN,NaN,NaN,0.0
1366,MI0062841,MEDILODGE OF STERLING,NaN,NaN,3.0,NaN,NaN,3.0
1368,MI0062955,MAJESTIC CARE OF FLUSHING,NaN,NaN,NaN,NaN,NaN,0.0


In [31]:
fixable_wrong_total = lead_service_line_inventory[
    lead_service_line_inventory["lead_lines"].notna() &
    lead_service_line_inventory["gpcl_lines"].notna() &
    lead_service_line_inventory["non_lead_lines"].notna() &
    lead_service_line_inventory["unknown_lines"].notna() &
    lead_service_line_inventory["total_lines"].notna() &
    (lead_service_line_inventory["calc_total"] != lead_service_line_inventory["total_lines"])
]

fixable_wrong_total

,pwsid,supply_name,lead_lines,gpcl_lines,non_lead_lines,unknown_lines,total_lines,calc_total
138,MI0001015,BUTTERCUP SHORES,0.0,0.0,26.0,0.0,70.0,26.0
374,MI0002790,GRAND RAPIDS,19430.0,0.0,60142.0,394.0,79963.0,79966.0
429,MI0003130,HESPERIA,187.0,46.0,36.0,83.0,410.0,352.0
479,MI0003430,IRONWOOD TOWNSHIP,0.0,20.0,383.0,7.0,372.0,410.0
611,MI0004240,MENDON,96.0,18.0,240.0,0.0,357.0,354.0
783,MI0005580,QUINCY,0.0,50.0,697.0,18.0,766.0,765.0
806,MI0005740,ROCKLAND TOWNSHIP,15.0,45.0,76.0,3.0,142.0,139.0
946,MI0006650,TRENTON,155.0,4.0,6730.0,0.0,6885.0,6889.0
1021,MI0007230,"YALE, CITY OF",0.0,0.0,732.0,38.0,743.0,770.0


In [32]:
lead_service_line_inventory["total_mismatch"] = (
    lead_service_line_inventory["calc_total"]
    != lead_service_line_inventory["total_lines"]
)

In [33]:
inventory_clean = lead_service_line_inventory.copy()

Mark as incomplete if there are any NaN value

In [34]:
inventory_clean["inventory_status"] = "complete"

mask_incomplete = inventory_clean[
    ["lead_lines", "gpcl_lines", "non_lead_lines", "unknown_lines", "total_lines"]
].isna().any(axis=1)

inventory_clean.loc[mask_incomplete, "inventory_status"] = "incomplete"

Mark as mismatch(True) if the total line does not match with calc_total

In [35]:
inventory_clean["total_mismatch"] = (
    inventory_clean["calc_total"] != inventory_clean["total_lines"]
)

In [36]:
inventory_clean.head()

,pwsid,supply_name,lead_lines,gpcl_lines,non_lead_lines,unknown_lines,total_lines,calc_total,total_mismatch,inventory_status
0,MI0000011,ACME TOWNSHIP - HOPE VILLAGE,0.0,0.0,1.0,0.0,1.0,1.0,False,complete
1,MI0000012,ADA TOWNSHIP,0.0,0.0,2431.0,0.0,2431.0,2431.0,False,complete
2,MI0000020,ADAMS TOWNSHIP,0.0,0.0,481.0,154.0,635.0,635.0,False,complete
3,MI0000030,ADDISON,0.0,0.0,118.0,34.0,152.0,152.0,False,complete
4,MI0000040,ADRIAN,5.0,2280.0,3876.0,0.0,6161.0,6161.0,False,complete


In [37]:
inventory_clean.to_csv(
    "clean_inventory.csv",
    index=False
)